In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

In [ ]:
data=pd.read_csv("../data/raw-data.csv")
data

In [ ]:
data.info()

## Preparing for visualization

In [ ]:
X=data.drop('Churn',axis=1)
y=data['Churn']

# Drop unnecessary ID column
X=X.drop('customerID',axis=1)

In [ ]:
# Senior citizen is a categorical column, converted to int dtype
# Convert it temporary back to 'Yes'/'No' for visualizations

X['SeniorCitizen']=data['SeniorCitizen'].apply(lambda x : 'Yes' if x == 1 else 'No')

In [ ]:
# TotalCharges has object dtype and cannot be converted to float because of ' ' elements
print( (X['TotalCharges']==' ').sum() )

# At this moment, drop all rows with wrong TotalCharges value
X=X[X['TotalCharges']!=' ']
y=y[X.index]

X['TotalCharges']=X['TotalCharges'].astype(float)

In [ ]:
# First step of cleaning: convert y to number representation
y_cleaned=y.apply(lambda x : 1 if x == 'Yes' else 0)

## Visualizations

In [ ]:
# Target distribution
global_ratio = y_cleaned.mean()

sns.countplot(x=y_cleaned,color='r')
plt.title(f'Target distribution. Ratio = {global_ratio.round(3)}')
plt.show()

In [ ]:
cat_cols=X.columns[X.dtypes == object]
num_cols=X.select_dtypes(include=['number']).columns

In [ ]:
# Visualize how data is distributed
num_cols2plot=4

fig,ax=plt.subplots(np.ceil(len(cat_cols)/num_cols2plot).astype(int), num_cols2plot,figsize=(18,12))

for i,col in enumerate(cat_cols):
    current_ax=ax[i // num_cols2plot][i % num_cols2plot]
    sns.countplot(x=X[col],ax=current_ax)
    if col == 'PaymentMethod':
        current_ax.tick_params(axis='x',rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize ratios of churn

fig, ax = plt.subplots(np.ceil(len(cat_cols)/num_cols2plot).astype(int), num_cols2plot, figsize=(16,18))

for i, col in enumerate(cat_cols):
    ratios = {}
    for val in X[col].unique():
        ratio = y_cleaned[X[col] == val].mean()
        ratios[val] = ratio

    current_ax = ax[i // num_cols2plot][i % num_cols2plot]
    sns.barplot(x=list(ratios.keys()), y=list(ratios.values()), ax=current_ax, color='b')
    current_ax.set_title(col)
    current_ax.set_ylim(0, 0.5)
    current_ax.axhline(global_ratio, color='k') # Baseline
    if col == 'PaymentMethod':
        current_ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize the only numerical features
fig,ax=plt.subplots(2,len(num_cols),figsize=(16,10))

for i,col in enumerate(num_cols):
    sns.boxplot(x=y, y=X[col],ax=ax[0][i],color='c')
    ax[0][i].set_title(col)
    ax[0][i].set_xlabel(None)
    ax[0][i].set_ylabel(None)

for i,col in enumerate(num_cols):
    sns.histplot(x=X[col],ax=ax[1][i],color='y',hue=y)
    ax[1][i].set_title(col)
    ax[1][i].set_xlabel(None)
    ax[1][i].set_ylabel('Count')

plt.show()

## Cleaning

In [ ]:
# Reset indexes to avoid wrong broadcasting

X=X.reset_index(drop=True)
y_cleaned=y_cleaned.reset_index(drop=True)

In [ ]:
# Apply onehot encoding to all categorical features and z-transformation to all numerical

encoder=OneHotEncoder(drop='first',sparse_output=False)
normalizer=StandardScaler()

preprocessor=ColumnTransformer([('enc',encoder,cat_cols),('norm',normalizer,num_cols)],remainder='passthrough')
preprocessor

In [ ]:
X_encoded=preprocessor.fit_transform(X)
X_cleaned=pd.DataFrame(X_encoded,columns=preprocessor.get_feature_names_out())
X_cleaned

In [ ]:
data_cleaned=pd.concat([X_cleaned,y_cleaned],axis=1)
data_cleaned